# Paper: ECG-Age for Cardiovascular Outcome Prediction: Evaluation in an Adult Congenital Heart Disease Cohort
## Merge ECG features and add demographic features to the feature set
### Seyedeh Somayyeh Mousavi
### Sep 2026
### Emory University
### bmemousavi@gmail.com¶

In [22]:
import pandas as pd
import os
import numpy as np
import shutil
import glob
import random
import warnings
warnings.filterwarnings("ignore")

In [23]:
input_path1 = '../../features/leadwise_features/'
input_path2 = '../../features/multileads_features/'
input_path3 = '../../model_and_metadata/'
input_path4 = '../../sample_data/'
output_path = '../../model_and_metadata/'

In [24]:
file_demo = 'demographics.csv'
mapping_label_demo = {'Female': 0, 'Male': 1}

# Count Number of .CSV files

In [25]:
csv_files_leadwise = glob.glob(os.path.join(input_path1, '**', '*.csv'),recursive=True)
print(f"Total lead-wise feature CSV files files in output: {len(csv_files_leadwise)}")
csv_files_multilead = glob.glob(os.path.join(input_path2, '**', '*.csv'),recursive=True)
print(f"Total multilead feature CSV files files in output: {len(csv_files_multilead)}")

Total lead-wise feature CSV files files in output: 5
Total multilead feature CSV files files in output: 5


# First two rows as header

In [26]:
header_lines_leadwise = pd.read_csv(csv_files_leadwise[0]).iloc[0:2, :]   
header_lines_leadwise    

,Features,ECG_len,n_beats,rmssd,sdnn,hr_median,hr_mean,hr_upper_5,hr_lower_5,hrf_pip,...,mean_ecg_beat_sample_96,mean_ecg_beat_sample_97,mean_ecg_beat_sample_98,mean_ecg_beat_sample_99,mean_ecg_beat_sample_100,svd1,svd2,svd3,svd4,svd5
0,Descriptions,ECG length,Number of ECG beats,Root Mean Square of Successive Differences for...,Standard Deviation of Normal-to-Normal Intervals,Median heart-rate,Mean heart-rate,95% heart-rate,5% heart-rate,Heart rate fragmentation: Percentage of inflec...,...,Average ECG beat sample 96,Average ECG beat sample 97,Average ECG beat sample 98,Average ECG beat sample 99,Average ECG beat sample 100,Normalized SVD value 1,Normalized SVD value 2,Normalized SVD value 3,Normalized SVD value 4,Normalized SVD value 5
1,Units,s,scalar,ms,ms,bpm,bpm,bpm,bpm,scalar,...,uv,uv,uv,uv,uv,scalar,scalar,scalar,scalar,scalar


In [27]:
header_csv_files_multilead = pd.read_csv(csv_files_multilead[0]).iloc[0:2, :]   
header_csv_files_multilead 

,Features,Taxis_mean.Form_1,Taxis_mean.Form_2,Taxis_mean.Form_3,Paxis_mean.Form_1,Paxis_mean.Form_2,Paxis_mean.Form_3,Raxis_mean.Form_1,Raxis_mean.Form_2,Raxis_mean.Form_3,Taxis_median.Form_1,Taxis_median.Form_2,Taxis_median.Form_3,Paxis_median.Form_1,Paxis_median.Form_2,Paxis_median.Form_3,Raxis_median.Form_1,Raxis_median.Form_2,Raxis_median.Form_3
0,Descriptions,"T-axis using mean amplitude: atan2(aVF, I)","T-axis using mean amplitude: atan2(2*aVF, sqrt...","T-axis using mean amplitude: atan2(2*II-I, sqr...","P-axis using mean amplitude: atan2(aVF, I)","P-axis using mean amplitude: atan2(2*aVF, sqrt...","P-axis using mean amplitude: atan2(2*II-I, sqr...","R-axis using mean amplitude: atan2(aVF, I)","R-axis using mean amplitude: atan2(2*aVF, sqrt...","R-axis using mean amplitude: atan2(2*II-I, sqr...","T-axis using median amplitude: atan2(aVF, I)","T-axis using median amplitude: atan2(2*aVF, sq...","T-axis using median amplitude: atan2(2*II-I, s...","P-axis using median amplitude: atan2(aVF, I)","P-axis using median amplitude: atan2(2*aVF, sq...","P-axis using median amplitude: atan2(2*II-I, s...","R-axis using median amplitude: atan2(aVF, I)","R-axis using median amplitude: atan2(2*aVF, sq...","R-axis using median amplitude: atan2(2*II-I, s..."
1,Units,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree,degree


# Process all lead-wise feature CSV files and combine them

In [28]:
df_list_leadwise = []
for file in csv_files_leadwise:
    
    # Skip the first two rows (header)
    df = pd.read_csv(file, skiprows=[1, 2])
    # add filename column
    df["source_file"] = os.path.basename(file).replace("_features_leadwise.csv", "") 
    df_list_leadwise.append(df)

# Concatenate all data
combined_df_leadwise = pd.concat(df_list_leadwise, ignore_index=True)
combined_df_leadwise.head(12)

,Features,ECG_len,n_beats,rmssd,sdnn,hr_median,hr_mean,hr_upper_5,hr_lower_5,hrf_pip,...,mean_ecg_beat_sample_97,mean_ecg_beat_sample_98,mean_ecg_beat_sample_99,mean_ecg_beat_sample_100,svd1,svd2,svd3,svd4,svd5,source_file
0,I,10,20,128.607,80.785,114.07,120.52,163.75,91.39,0.632,...,38.592,48.960,37.825,33.929,21.268,10.776,7.153,6.640,5.181,JS00001
1,II,10,20,77.569,50.561,113.21,118.93,148.77,104.92,0.737,...,-19.354,-19.777,-22.076,-18.492,31.710,12.297,7.466,5.382,4.202,JS00001
2,III,10,20,77.593,50.956,113.21,119.06,149.41,105.05,0.737,...,-74.097,-57.778,-61.277,-59.165,27.905,11.830,9.383,7.752,5.320,JS00001
3,aVR,10,20,77.909,50.140,113.64,118.89,149.05,105.11,0.789,...,-13.135,-15.126,-13.340,-7.623,27.122,10.867,7.294,5.829,4.871,JS00001
4,aVL,10,20,122.521,83.429,114.07,121.03,166.08,93.97,0.789,...,36.214,28.929,23.324,23.956,19.543,11.996,8.776,6.955,6.755,JS00001
5,aVF,10,20,77.713,50.868,114.50,119.05,149.05,104.26,0.737,...,-44.371,-39.625,-44.413,-37.173,31.768,12.658,8.764,6.258,5.115,JS00001
6,V1,10,20,88.396,56.799,114.94,119.24,149.05,99.60,0.842,...,-60.953,-49.869,-39.570,-24.561,39.662,11.776,8.772,7.208,5.556,JS00001
7,V2,10,20,82.137,52.238,112.36,119.00,148.87,104.21,0.737,...,29.401,22.141,16.529,10.279,51.159,8.454,6.428,4.976,4.491,JS00001
8,V3,10,20,80.788,52.156,113.21,119.05,148.89,104.21,0.737,...,65.085,49.385,42.211,30.079,48.850,11.015,7.507,5.851,4.494,JS00001
9,V4,10,20,77.487,50.350,113.21,118.87,148.31,104.38,0.789,...,94.569,76.482,60.859,42.808,43.520,15.548,9.792,8.581,5.084,JS00001


# Process all multilead feature CSV files and combine them

In [29]:
df_list_multilead = []
for file in csv_files_multilead:
    
    # Skip the first two rows (header)
    df = pd.read_csv(file, skiprows=[1, 2])
    # add filename column
    df["source_file"] = os.path.basename(file).replace("_features_multileads.csv", "") 
    df_list_multilead.append(df)

# Concatenate all data
combined_df_multilead = pd.concat(df_list_multilead, ignore_index=True)
combined_df_multilead = combined_df_multilead.drop(columns=['Features'])
combined_df_multilead.head(12)

,Taxis_mean.Form_1,Taxis_mean.Form_2,Taxis_mean.Form_3,Paxis_mean.Form_1,Paxis_mean.Form_2,Paxis_mean.Form_3,Raxis_mean.Form_1,Raxis_mean.Form_2,Raxis_mean.Form_3,Taxis_median.Form_1,Taxis_median.Form_2,Taxis_median.Form_3,Paxis_median.Form_1,Paxis_median.Form_2,Paxis_median.Form_3,Raxis_median.Form_1,Raxis_median.Form_2,Raxis_median.Form_3,source_file
0,-33,-37,8,30,34,-29,70,72,72,-32,-35,2,22,25,-29,70,73,73,JS00001
1,29,32,31,29,33,36,39,43,42,29,32,32,29,33,37,38,42,41,JS00002
2,5,5,-1,29,33,40,24,27,27,4,5,-1,32,36,39,24,27,27,JS00004
3,26,30,20,134,130,111,33,37,37,25,29,17,135,131,113,33,37,37,JS00005
4,16,18,17,20,23,18,22,25,27,15,18,18,21,24,20,22,25,27,JS00006


# Combine df_multilead and df_leadwise 
## (General_application)

In [30]:
combined_df_leadwise1 = combined_df_leadwise.drop(columns= ['ECG_len', 'n_beats'])
mean_ecg_cols = [
    c for c in combined_df_leadwise1.columns
    if c.startswith('mean_ecg_beat_sample_')]   

df_non_mean_ecg = combined_df_leadwise1.drop(columns=mean_ecg_cols)
ecg_feature_wide1 = df_non_mean_ecg.pivot(index=['source_file'], columns='Features')
ecg_feature_wide1.columns = [f"{feat}_{src}" for feat, src in ecg_feature_wide1.columns]
ecg_feature_wide1 = ecg_feature_wide1.reset_index()

df_mean_ecg = combined_df_leadwise1[['Features', 'source_file'] + mean_ecg_cols ]
ecg_feature_wide2 = df_mean_ecg.pivot(index=['source_file'], columns='Features')
ecg_feature_wide2.columns = [f"{feat}_{src}" for feat, src in ecg_feature_wide2.columns]
ecg_feature_wide2 = ecg_feature_wide2.reset_index()

ecg_feature_leadwise = pd.merge(ecg_feature_wide1, ecg_feature_wide2, on=['source_file'])
cols_to_convert = [col for col in ecg_feature_leadwise.columns if col not in ['source_file']]
ecg_feature_leadwise[cols_to_convert] = ecg_feature_leadwise[cols_to_convert].apply(pd.to_numeric, errors='coerce')
ecg_feature = pd.merge(ecg_feature_leadwise, combined_df_multilead, on = 'source_file')
ecg_feature

,source_file,rmssd_I,rmssd_II,rmssd_III,rmssd_V1,rmssd_V2,rmssd_V3,rmssd_V4,rmssd_V5,rmssd_V6,...,Raxis_mean.Form_3,Taxis_median.Form_1,Taxis_median.Form_2,Taxis_median.Form_3,Paxis_median.Form_1,Paxis_median.Form_2,Paxis_median.Form_3,Raxis_median.Form_1,Raxis_median.Form_2,Raxis_median.Form_3
0,JS00001,128.607,77.569,77.593,88.396,82.137,80.788,77.487,77.110,159.834,...,72,-32,-35,2,22,25,-29,70,73,73
1,JS00002,50.544,47.847,49.396,44.400,47.596,47.589,49.746,48.744,48.744,...,42,29,32,32,29,33,37,38,42,41
2,JS00004,118.956,25.117,144.561,25.824,25.724,122.917,23.713,25.219,26.436,...,27,4,5,-1,32,36,39,24,27,27
3,JS00005,6.028,5.557,5.477,4.601,5.370,5.600,5.672,5.769,5.810,...,37,25,29,17,135,131,113,33,37,37
4,JS00006,231.098,38.083,266.131,308.201,151.889,151.949,175.542,38.826,39.141,...,27,15,18,18,21,24,20,22,25,27


# Combine df_multilead and df_leadwise 
## (CinC2026, ECG-Age model)
## The latest version of the OSET ECG feature extraction toolbox provides additional features, such as cardiac axis features. These are **not** model inputs and should be removed before running the prediction.

In [31]:
combined_df_leadwise2 = combined_df_leadwise.drop(columns= ['ECG_len', 'n_beats'])
mean_ecg_cols = [
    c for c in combined_df_leadwise1.columns
    if c.startswith('mean_ecg_beat_sample_')]

df_non_mean_ecg = combined_df_leadwise2.drop(columns=mean_ecg_cols)
df_non_mean_ecg = df_non_mean_ecg.rename(columns={'ratio_qrs_t_amp': 'ratio_t_r_amp', 'ratio_qrs_p_amp': 'ratio_p_r_amp'})
ecg_feature_wide1 = df_non_mean_ecg.pivot(index=['source_file'], columns='Features')
ecg_feature_wide1.columns = [f"{feat}_{src}" for feat, src in ecg_feature_wide1.columns]
ecg_feature_wide1 = ecg_feature_wide1.reset_index()

df_mean_ecg = combined_df_leadwise1[['Features', 'source_file'] + mean_ecg_cols ]
df_mean_ecg = df_mean_ecg[df_mean_ecg['Features'].isin(['I', 'II'])]
ecg_feature_wide2 = df_mean_ecg.pivot(index=['source_file'], columns='Features')
ecg_feature_wide2.columns = [f"{feat}_{src}" for feat, src in ecg_feature_wide2.columns]
ecg_feature_wide2 = ecg_feature_wide2.reset_index()

ecg_feature_leadwise = pd.merge(ecg_feature_wide1, ecg_feature_wide2, on=['source_file'])
cols_to_convert = [col for col in ecg_feature_leadwise.columns if col not in ['source_file']]
ecg_feature_leadwise[cols_to_convert] = ecg_feature_leadwise[cols_to_convert].apply(pd.to_numeric, errors='coerce')

df_demo = pd.read_csv(input_path4 + file_demo)
ecg_feature_CinC2026 = pd.merge(ecg_feature_leadwise, df_demo, on=['source_file'])
ecg_feature_CinC2026['GENDER'] = ecg_feature_CinC2026['GENDER'].map(mapping_label_demo)
ecg_feature_CinC2026 = ecg_feature_CinC2026.rename(columns={'source_file': 'FileID'})
ecg_feature_CinC2026

,FileID,rmssd_I,rmssd_II,rmssd_III,rmssd_V1,rmssd_V2,rmssd_V3,rmssd_V4,rmssd_V5,rmssd_V6,...,mean_ecg_beat_sample_97_I,mean_ecg_beat_sample_97_II,mean_ecg_beat_sample_98_I,mean_ecg_beat_sample_98_II,mean_ecg_beat_sample_99_I,mean_ecg_beat_sample_99_II,mean_ecg_beat_sample_100_I,mean_ecg_beat_sample_100_II,GENDER,Age
0,JS00001,128.607,77.569,77.593,88.396,82.137,80.788,77.487,77.110,159.834,...,38.592,-19.354,48.960,-19.777,37.825,-22.076,33.929,-18.492,1,85
1,JS00002,50.544,47.847,49.396,44.400,47.596,47.589,49.746,48.744,48.744,...,-2.175,-18.978,-4.481,-20.466,-4.946,-25.976,-6.586,-30.508,0,59
2,JS00004,118.956,25.117,144.561,25.824,25.724,122.917,23.713,25.219,26.436,...,-53.561,-39.123,-50.556,-33.478,-52.909,-32.633,-51.723,-35.153,1,66
3,JS00005,6.028,5.557,5.477,4.601,5.370,5.600,5.672,5.769,5.810,...,60.909,42.240,68.406,50.276,67.687,53.302,70.475,49.468,0,73
4,JS00006,231.098,38.083,266.131,308.201,151.889,151.949,175.542,38.826,39.141,...,-36.699,-20.927,-41.262,-23.040,-36.028,-24.704,-33.170,-24.331,0,46


# Save the feature set

In [32]:
output_file_name = 'inference_mode_pretrained_model_outputs_cinc2026.csv'
output_file_path = os.path.join(output_path, output_file_name)
ecg_feature_CinC2026.to_csv(output_file_path, index=False)